# LED Manager Smoke Tests

Staged tests for `LedSource` and `LedManager`. Virtual hardware is the safe default. For physical hardware, configure one controller and LED source below, verify the selected channel, then explicitly enable the guarded illumination cells.

In [8]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
import time


def find_repo_root() -> Path:
    """Return the repository root containing the evomachine package."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "evomachine" / "peripherals" / "leds.py").is_file():
            return candidate
    raise RuntimeError("Could not find the EvoMachine repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evomachine.bindings.binding_types import BindingType
from evomachine.peripherals.leds import LedConfig, LedFactory, LedManager
from evomachine.peripherals.peripheralcontrollers import (
    PeripheralControllerConfig,
    PeripheralControllerFactory,
    SerialPeripheralControllerConfig,
)
from evomachine.types import LEDType


@dataclass()
class LedManagerTestSettings:
    binding: BindingType 
    available_leds: tuple[LEDType, ...]
    test_led: LEDType 
    test_brightness: float
    test_duration_ms: float 
    port: str | None = None
    hwid: str | None = None

syncboard_test_settings = LedManagerTestSettings(
    binding=BindingType.SYNCBOARD,
    available_leds=(LEDType.LED_385_NM, LEDType.LED_450_NM, LEDType.LED_515_NM, LEDType.LED_565_NM, LEDType.LED_645_NM),
    test_led=LEDType.LED_450_NM,
    test_brightness=100.0,
    test_duration_ms=100.0,
    hwid = "USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0",
)
asi_tiger_test_settings = LedManagerTestSettings(
    binding=BindingType.ASI_TIGER,
    available_leds=(LEDType.LED_OVERHEAD_TIGER,),
    test_led=LEDType.LED_OVERHEAD_TIGER,
    test_brightness=100.0,
    test_duration_ms=100.0,
    hwid = "USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3",
)
#Not sure this is connected
kwr103_test_settings = LedManagerTestSettings(
    binding=BindingType.KWR103,
    available_leds=(LEDType.LED_OVERHEAD,),
    test_led=LEDType.LED_OVERHEAD,
    test_brightness=100.0,
    test_duration_ms=100.0
    
)

# Physical examples:
# SyncBoard: binding=BindingType.SYNCBOARD, available_leds=(LEDType.LED_385_NM,LEDType.LED_450_NM,LEDType.LED_515_NM,LEDType.LED_565_NM,LEDType.LED_645_NM,), hwid="USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0"
# ASI Tiger: binding=BindingType.ASI_TIGER, available_leds=(LEDType.LED_OVERHEAD_TIGER,), hwid="USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3"
# KWR103: binding=BindingType.KWR103, available_leds=(LEDType.LED_OVERHEAD,), port="/dev/ttyUSB0"

SETTINGS = asi_tiger_test_settings  # Change this to select the desired LED manager configuration.

# This must be changed deliberately before either illumination cell will run.
RUN_LED_TEST = True
SETTINGS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LedManagerTestSettings(binding=<BindingType.SYNCBOARD: 3>, available_leds=(<LEDType.LED_385_NM: 0>, <LEDType.LED_450_NM: 1>, <LEDType.LED_515_NM: 2>, <LEDType.LED_565_NM: 3>, <LEDType.LED_645_NM: 4>), test_led=<LEDType.LED_450_NM: 1>, test_brightness=100.0, test_duration_ms=100.0, port=None, hwid='USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0')

## Inspect serial ports

Use this to identify a physical controller. Set exactly one of `port` or `hwid`; neither is needed for the virtual binding.

In [14]:
try:
    from serial.tools import list_ports
except ImportError as error:
    raise RuntimeError("pyserial is required to inspect serial ports.") from error

[
    {"device": port.device, "description": port.description, "hwid": port.hwid}
    for port in list_ports.comports()
]

[{'device': '/dev/ttyS1', 'description': 'ttyS1', 'hwid': 'PNP0501'},
 {'device': '/dev/ttyS0', 'description': 'ttyS0', 'hwid': 'PNP0501'},
 {'device': '/dev/ttyUSB0',
  'description': 'CP2102 USB to UART Bridge Controller - CP2102 USB to UART Bridge Controller',
  'hwid': 'USB VID:PID=10C4:EA60 SER=0001 LOCATION=5-3'},
 {'device': '/dev/ttyACM0',
  'description': 'USB Serial',
  'hwid': 'USB VID:PID=16C0:0483 SER=14582700 LOCATION=7-2:1.0'}]

## Create the controller, source, and manager

Initialisation should leave every LED disabled. Re-running this cell cleans up objects from its previous run first.

In [9]:
previous_manager = globals().get("led_manager")
if previous_manager is not None and previous_manager.is_initialised():
    previous_manager.disable_led()
    previous_manager.finalise()
previous_controller = globals().get("led_controller")
if previous_controller is not None and previous_controller.is_initialised():
    previous_controller.shutdown()

if SETTINGS.binding == BindingType.VIRTUAL:
    controller_config = PeripheralControllerConfig(binding=BindingType.VIRTUAL)
elif SETTINGS.binding in {BindingType.SYNCBOARD, BindingType.ASI_TIGER, BindingType.KWR103}:
    controller_config = SerialPeripheralControllerConfig(
        binding=SETTINGS.binding,
        port=SETTINGS.port,
        hwid=SETTINGS.hwid,
    )
else:
    raise ValueError(f"Unsupported LED binding: {SETTINGS.binding}")

led_controller = PeripheralControllerFactory.create(controller_config)
led_source = LedFactory.create(
    LedConfig(
        binding=SETTINGS.binding,
        available_leds=list(SETTINGS.available_leds),
    ),
    peripheral_controllers=led_controller,
)
led_manager = LedManager([led_source])
led_manager.initialise()
led_manager.disable_led()

{
    "manager": led_manager.name,
    "source": led_source.name,
    "binding": SETTINGS.binding,
    "controller_initialised": led_controller.is_initialised(),
    "manager_initialised": led_manager.is_initialised(),
    "manager_alive": led_manager.is_alive(),
    "available_leds": led_manager.get_available_leds(),
}

2026-07-02 09:47:53 - DEBUG - syncboard.serialconnection - Connecting to /dev/ttyACM0 at 2000000 baud
2026-07-02 09:47:53 - DEBUG - syncboard.command - Formatted command: $attachLED/true#%
2026-07-02 09:47:53 - DEBUG - syncboard.syncboardcontroller - Sending command $attachLED/true#% at attempt 0.
2026-07-02 09:47:53 - DEBUG - syncboard.serialconnection - Sending data: b'$attachLED/true#%'
2026-07-02 09:47:53 - DEBUG - syncboard.serialconnection - Received: $attachLED/1#%

2026-07-02 09:47:53 - DEBUG - syncboard.serialconnection - Received: ['$attachLED/1#%']
2026-07-02 09:47:53 - INFO - syncboard.serialconnection - SyncBoardController.SerialConnection: syncboard responded with ['$attachLED/1#%'] to $attachLED/true#%.
2026-07-02 09:47:54 - DEBUG - syncboard.serialconnection - Received: 
2026-07-02 09:47:54 - DEBUG - syncboard.command - Formatted command: $attachMagnet/true#%
2026-07-02 09:47:54 - DEBUG - syncboard.syncboardcontroller - Sending command $attachMagnet/true#% at attempt 0.

{'manager': 'LED Manager',
 'source': 'SyncBoard LED Source',
 'binding': <BindingType.SYNCBOARD: 3>,
 'controller_initialised': True,
 'manager_initialised': True,
 'manager_alive': True,
 'available_leds': [<LEDType.LED_385_NM: 0>,
  <LEDType.LED_450_NM: 1>,
  <LEDType.LED_515_NM: 2>,
  <LEDType.LED_565_NM: 3>,
  <LEDType.LED_645_NM: 4>]}

## Inspect cached LED states

In [3]:
if "led_manager" not in globals() or not led_manager.is_initialised():
    raise RuntimeError("Run the manager creation cell first.")

[led_manager.get_led_state(led_type) for led_type in led_manager.get_available_leds()]

[LedState(led_type=<LEDType.LED_385_NM: 0>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_450_NM: 1>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_515_NM: 2>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_565_NM: 3>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_645_NM: 4>, brightness=0.0, is_on=False, stop_time=None)]

## Test one timed LED pulse

Confirm that `SETTINGS.test_led` identifies the intended physical channel. The `finally` block disables all managed LEDs even if the test raises an exception.

In [10]:
if not RUN_LED_TEST:
    raise RuntimeError("Set RUN_LED_TEST = True in the setup cell after checking the hardware configuration.")
if SETTINGS.test_led not in led_manager.get_available_leds():
    raise ValueError(f"{SETTINGS.test_led} is not managed by this LED manager.")

try:
    led_manager.set_led(
        led_type=SETTINGS.test_led,
        brightness=SETTINGS.test_brightness,
        duration=SETTINGS.test_duration_ms*10,
    )
    state_during_pulse = led_manager.get_led_state(SETTINGS.test_led)
finally:
    led_manager.disable_led()

{
    "during_pulse": state_during_pulse,
    "after_disable": led_manager.get_led_state(SETTINGS.test_led),
}

2026-07-02 09:48:26 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_450_NM brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 09:48:26 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_450_NM to brightness=100.0 duration=1000.0.
2026-07-02 09:48:26 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 09:48:26 - DEBUG - syncboard.command - Formatted command: $setupLED/5/0/1.0000000#%
2026-07-02 09:48:26 - DEBUG - syncboard.syncboardcontroller - Sending command $setupLED/5/0/1.0000000#% at attempt 0.
2026-07-02 09:48:26 - DEBUG - syncboard.serialconnection - Sending data: b'$setupLED/5/0/1.0000000#%'
2026-07-02 09:48:26 - DEBUG - syncboard.serialconnection - Received: $Received setup LED command/1#%

2026-07-02 09:48:26 - DEBUG - syncboard.serialconnection - Received: $setupLED/5#%

2026-07-02 09:48:26 - DEBUG - syncboard.serialconnection - Received

{'during_pulse': LedState(led_type=<LEDType.LED_450_NM: 1>, brightness=100.0, is_on=True, stop_time=1782982109.1455479),
 'after_disable': LedState(led_type=<LEDType.LED_450_NM: 1>, brightness=0.0, is_on=False, stop_time=None)}

## Test manager routing across all configured LEDs

Each channel receives one short pulse in sequence. Skip this cell unless every configured channel is safe to illuminate.

In [12]:
if not RUN_LED_TEST:
    raise RuntimeError("Set RUN_LED_TEST = True in the setup cell after checking the hardware configuration.")

try:
    for led_type in led_manager.get_available_leds():
        print(f"Pulsing {led_type.name}")
        led_manager.set_led(
            led_type=led_type,
            brightness=SETTINGS.test_brightness,
            duration=SETTINGS.test_duration_ms*10,
        )
        #time.sleep(SETTINGS.test_duration_ms / 1000.0 + 0.1)
finally:
    led_manager.disable_led()

[led_manager.get_led_state(led_type) for led_type in led_manager.get_available_leds()]

2026-07-02 09:48:57 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_385_NM brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 09:48:57 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_385_NM to brightness=100.0 duration=1000.0.
2026-07-02 09:48:57 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 09:48:57 - DEBUG - syncboard.syncboardcontroller - LED LED_ID.LED_385_NM already configured: {'mode': 0, 'intensity': 1.0, 'status': 'off', 'stop_time': 1782982123.3549848}
2026-07-02 09:48:57 - DEBUG - syncboard.command - Formatted command: $switchLEDTimed/7/1000.0000000#%
2026-07-02 09:48:57 - DEBUG - syncboard.syncboardcontroller - Sending command $switchLEDTimed/7/1000.0000000#% at attempt 0.
2026-07-02 09:48:57 - DEBUG - syncboard.serialconnection - Sending data: b'$switchLEDTimed/7/1000.0000000#%'
2026-07-02 09:48:57 - DEBUG - syncboard.serialco

Pulsing LED_385_NM


2026-07-02 09:48:58 - DEBUG - syncboard.serialconnection - Received: Turning LED 7 off after 1000008 microseconds.

2026-07-02 09:48:58 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_450_NM brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 09:48:58 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_450_NM to brightness=100.0 duration=1000.0.
2026-07-02 09:48:58 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 09:48:58 - DEBUG - syncboard.syncboardcontroller - LED LED_ID.LED_450_NM already configured: {'mode': 0, 'intensity': 1.0, 'status': 'off', 'stop_time': 1782982124.3617864}
2026-07-02 09:48:58 - DEBUG - syncboard.command - Formatted command: $switchLEDTimed/5/1000.0000000#%
2026-07-02 09:48:58 - DEBUG - syncboard.syncboardcontroller - Sending command $switchLEDTimed/5/1000.0000000#% at attempt 0.
2026-07-02 09:48:58 - DEBUG - syncboard.s

Pulsing LED_450_NM


2026-07-02 09:48:59 - DEBUG - syncboard.serialconnection - Received: Turning LED 5 off after 1000008 microseconds.

2026-07-02 09:48:59 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_515_NM brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 09:48:59 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_515_NM to brightness=100.0 duration=1000.0.
2026-07-02 09:48:59 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 09:48:59 - DEBUG - syncboard.syncboardcontroller - LED LED_ID.LED_515_NM already configured: {'mode': 0, 'intensity': 1.0, 'status': 'off', 'stop_time': 1782982126.422641}
2026-07-02 09:48:59 - DEBUG - syncboard.command - Formatted command: $switchLEDTimed/2/1000.0000000#%
2026-07-02 09:48:59 - DEBUG - syncboard.syncboardcontroller - Sending command $switchLEDTimed/2/1000.0000000#% at attempt 0.
2026-07-02 09:48:59 - DEBUG - syncboard.se

...

Pulsing LED_565_NM


2026-07-02 09:49:01 - DEBUG - syncboard.serialconnection - Received: Turning LED 3 off after 1000008 microseconds.

2026-07-02 09:49:01 - DEBUG - evomachine.peripherals.leds - LedManager.set_led: routing LED_645_NM brightness=100.0 duration=1000.0 through LED Manager.
2026-07-02 09:49:01 - DEBUG - evomachine.peripherals.leds - LedSource.set_led: setting SyncBoard LED Source LED_645_NM to brightness=100.0 duration=1000.0.
2026-07-02 09:49:01 - WARNING - syncboard.syncboardcontroller - Received intensity 1.0 for enable_led. Duration is 1000.0.
2026-07-02 09:49:01 - DEBUG - syncboard.syncboardcontroller - LED LED_ID.LED_645_NM already configured: {'mode': 0, 'intensity': 1.0, 'status': 'off', 'stop_time': 1782982130.5426993}
2026-07-02 09:49:01 - DEBUG - syncboard.command - Formatted command: $switchLEDTimed/4/1000.0000000#%
2026-07-02 09:49:01 - DEBUG - syncboard.syncboardcontroller - Sending command $switchLEDTimed/4/1000.0000000#% at attempt 0.
2026-07-02 09:49:01 - DEBUG - syncboard.s

Pulsing LED_645_NM


2026-07-02 09:49:02 - DEBUG - syncboard.serialconnection - Received: Turning LED 4 off after 1000008 microseconds.

2026-07-02 09:49:02 - DEBUG - evomachine.peripherals.leds - LedManager.disable_led: disabling all LEDs for LED Manager.
2026-07-02 09:49:02 - DEBUG - evomachine.peripherals.leds - LedSource.disable_led: disabling LED_385_NM on SyncBoard LED Source.
2026-07-02 09:49:02 - DEBUG - syncboard.command - Formatted command: $switchLED/7/0#%
2026-07-02 09:49:02 - DEBUG - syncboard.syncboardcontroller - Sending command $switchLED/7/0#% at attempt 0.
2026-07-02 09:49:02 - DEBUG - syncboard.serialconnection - Sending data: b'$switchLED/7/0#%'
2026-07-02 09:49:02 - DEBUG - syncboard.serialconnection - Received: Called switchLEDDirect with args channel=7 on=0 force0

2026-07-02 09:49:02 - DEBUG - syncboard.serialconnection - Received: Resetting LED timeout for channel 7

2026-07-02 09:49:02 - DEBUG - syncboard.serialconnection - Received: LED is not timed. Cannot reset timeout.

2026-0

[LedState(led_type=<LEDType.LED_385_NM: 0>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_450_NM: 1>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_515_NM: 2>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_565_NM: 3>, brightness=0.0, is_on=False, stop_time=None),
 LedState(led_type=<LEDType.LED_645_NM: 4>, brightness=0.0, is_on=False, stop_time=None)]

## Cleanup

Always run this before unplugging or switching LED hardware.

In [13]:
if "led_manager" in globals() and led_manager is not None:
    led_manager.disable_led()
    if led_manager.is_initialised():
        led_manager.finalise()

if "led_controller" in globals() and led_controller is not None:
    if led_controller.is_initialised():
        led_controller.shutdown()

{
    "manager_initialised": led_manager.is_initialised(),
    "controller_initialised": led_controller.is_initialised(),
}

2026-07-02 09:49:20 - DEBUG - evomachine.peripherals.leds - LedManager.disable_led: disabling all LEDs for LED Manager.
2026-07-02 09:49:20 - DEBUG - evomachine.peripherals.leds - LedSource.disable_led: disabling LED_385_NM on SyncBoard LED Source.
2026-07-02 09:49:20 - DEBUG - syncboard.command - Formatted command: $switchLED/7/0#%
2026-07-02 09:49:20 - DEBUG - syncboard.syncboardcontroller - Sending command $switchLED/7/0#% at attempt 0.
2026-07-02 09:49:20 - DEBUG - syncboard.serialconnection - Sending data: b'$switchLED/7/0#%'
2026-07-02 09:49:20 - DEBUG - syncboard.serialconnection - Received: Called switchLEDDirect with args channel=7 on=0 force0

2026-07-02 09:49:20 - DEBUG - syncboard.serialconnection - Received: Resetting LED timeout for channel 7

2026-07-02 09:49:20 - DEBUG - syncboard.serialconnection - Received: LED is not timed. Cannot reset timeout.

2026-07-02 09:49:20 - DEBUG - syncboard.serialconnection - Received: $switchLED/7#%

2026-07-02 09:49:20 - DEBUG - syncboa

{'manager_initialised': False, 'controller_initialised': False}